In [1]:
pip install python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [23]:
import pandas as pd          
from scipy import stats 
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()  # this reads the .env file and loads it into memory

api_key = os.getenv("ANTHROPIC_API_KEY")  # grabs the key without ever typing it out

client = anthropic.Anthropic(api_key=api_key)

In [24]:

# Load the cleaned data from notebook 01


client_table = pd.read_csv("/Users/akashkumarsamantray/BI_Agent/bi-insight-agent/client_table_clean.csv")
web = pd.read_csv("/Users/akashkumarsamantray/BI_Agent/bi-insight-agent/web_events_clean.csv", parse_dates=['date_time'])

print(client_table.shape)
print(web.shape)

(50487, 10)
(317235, 6)


In [25]:

# Build a list of "segments" the agent is allowed to test

# This is just a list telling the agent what columns it's allowed to slice the data by.
# Instead of the agent randomly guessing, we give it a fixed toolbox to work with 
# same idea as giving a new analyst a list of filters they're allowed to use.

segments_to_check = {
    "age_group": "clnt_age",
    "tenure_group": "clnt_tenure_yr",
    "gender": "gendr",
    "num_accts": "num_accts"
}

print(segments_to_check)

{'age_group': 'clnt_age', 'tenure_group': 'clnt_tenure_yr', 'gender': 'gendr', 'num_accts': 'num_accts'}


In [26]:


# Rebuild the "completed" table (same logic as notebook 01/02)

step_order = {'start': 0, 'step_1': 1, 'step_2': 2, 'step_3': 3, 'confirm': 4}
web['step_rank'] = web['process_step'].map(step_order)

furthest_step = (
    web.groupby('client_id')['step_rank']
    .max()
    .reset_index()
    .rename(columns={'step_rank': 'furthest_step_rank'})
)

furthest_step = furthest_step.merge(client_table[['client_id', 'Variation']], on='client_id', how='left')
furthest_step['completed'] = furthest_step['furthest_step_rank'] == 4

print(furthest_step.head())

   client_id  furthest_step_rank Variation  completed
0        555                   4      Test       True
1        647                   4      Test       True
2        934                   0      Test      False
3       1028                   3   Control      False
4       1104                   0   Control      False


In [27]:

# Rebuild duration and backward-navigation data (same logic as notebook 02)


# Duration: how long each client took, per visit, then averaged across their visits
visit_duration = (
    web.groupby(['client_id', 'visit_id'])['date_time']
    .agg(['min', 'max'])
    .reset_index()
)
visit_duration['duration_sec'] = (visit_duration['max'] - visit_duration['min']).dt.total_seconds()
client_duration = visit_duration.groupby('client_id')['duration_sec'].mean().reset_index()

# Backward navigation: did the client ever move back a step (e.g. step_3 -> step_2)?
web_sorted = web.sort_values(['client_id', 'visit_id', 'date_time'])
web_sorted['prev_step_rank'] = web_sorted.groupby(['client_id', 'visit_id'])['step_rank'].shift(1)
web_sorted['went_backward'] = web_sorted['step_rank'] < web_sorted['prev_step_rank']
client_backward = web_sorted.groupby('client_id')['went_backward'].any().reset_index()

print(client_duration.shape)
print(client_backward.shape)


    

(50500, 2)
(50500, 2)


In [34]:


# The main test function: checks ONE segment against ONE metric

def test_one_segment(column_name, metric_column, client_table, metric_data, is_numeric_metric, min_group_size=30):
    """
    Tests whether Test vs Control differ, within one segment (like gender or age),
    for one metric (like completed, duration, or backward navigation).
    """

    # Step 1: bring the segment column into the metric data FIRST
    merged = metric_data.merge(
        client_table[['client_id', column_name]],
        on='client_id', how='left'
    )

    # Step 2: build the actual group (High/Low for numbers, or category as-is)
    if pd.api.types.is_numeric_dtype(merged[column_name]):
        midpoint = merged[column_name].median()
        merged['group'] = merged[column_name].apply(lambda x: "High" if x >= midpoint else "Low")
    else:
        merged['group'] = merged[column_name]

    # Step 3: NOW check sample size — on the actual group column, not the raw column
    group_counts = merged['group'].value_counts()
    if (group_counts < min_group_size).any():
        print(f"Skipping {column_name} — one or more groups too small to trust")
        print(group_counts)
        return None

    # Step 4: compare Test vs Control within each group
    result_table = merged.groupby(['group', 'Variation'])[metric_column].mean()

    # Step 5: run the right statistical test
    if is_numeric_metric:
        test_vals = merged.loc[merged['Variation'] == 'Test', metric_column].dropna()
        control_vals = merged.loc[merged['Variation'] == 'Control', metric_column].dropna()
        stat, p_value = stats.ttest_ind(test_vals, control_vals, equal_var=False)
    else:
        contingency = pd.crosstab(merged['Variation'], merged[metric_column])
        stat, p_value, dof, expected = stats.chi2_contingency(contingency)

    return {
        "segment": column_name,
        "metric": metric_column,
        "result_table": result_table,
        "p_value": p_value
    }

In [35]:


# Run the agent's investigation across ALL segments AND all 3 metrics

all_results = []

# attach Variation to duration and backward-navigation tables so we can compare Test vs Control
client_duration_full = client_duration.merge(client_table[['client_id', 'Variation']], on='client_id', how='left')
client_backward_full = client_backward.merge(client_table[['client_id', 'Variation']], on='client_id', how='left')

metrics_to_test = [
    {"data": furthest_step, "column": "completed", "is_numeric": False},
    {"data": client_duration_full, "column": "duration_sec", "is_numeric": True},
    {"data": client_backward_full, "column": "went_backward", "is_numeric": False},
]

for metric_info in metrics_to_test:
    for segment_name, column_name in segments_to_check.items():
        result = test_one_segment(
            column_name=column_name,
            metric_column=metric_info["column"],
            client_table=client_table,
            metric_data=metric_info["data"],
            is_numeric_metric=metric_info["is_numeric"]
        )
        if result is not None:
            all_results.append(result)

print(f"Total reliable results: {len(all_results)}")

Skipping gendr — one or more groups too small to trust
group
U    17280
M    16947
F    16258
X        2
Name: count, dtype: int64
Skipping num_accts — one or more groups too small to trust
group
High    50486
Low        14
Name: count, dtype: int64
Skipping gendr — one or more groups too small to trust
group
U    17280
M    16947
F    16258
X        2
Name: count, dtype: int64
Skipping num_accts — one or more groups too small to trust
group
High    50486
Low        14
Name: count, dtype: int64
Skipping gendr — one or more groups too small to trust
group
U    17280
M    16947
F    16258
X        2
Name: count, dtype: int64
Skipping num_accts — one or more groups too small to trust
group
High    50486
Low        14
Name: count, dtype: int64
Total reliable results: 6


In [36]:


# Turn the results into plain text so Claude can read them


findings_text = ""

for r in all_results:
    findings_text += f"\nSegment: {r['segment']}  |  Metric: {r['metric']}\n"
    findings_text += f"p-value: {r['p_value']:.5f}\n"
    findings_text += str(r['result_table'])
    findings_text += "\n"

print(findings_text)


Segment: clnt_age  |  Metric: completed
p-value: 0.00000
group  Variation
High   Control      0.638259
       Test         0.670604
Low    Control      0.674007
       Test         0.715185
Name: completed, dtype: float64

Segment: clnt_tenure_yr  |  Metric: completed
p-value: 0.00000
group  Variation
High   Control      0.650099
       Test         0.684638
Low    Control      0.661840
       Test         0.701427
Name: completed, dtype: float64

Segment: clnt_age  |  Metric: duration_sec
p-value: 0.00000
group  Variation
High   Control      333.854578
       Test         380.807579
Low    Control      252.082537
       Test         276.069382
Name: duration_sec, dtype: float64

Segment: clnt_tenure_yr  |  Metric: duration_sec
p-value: 0.00000
group  Variation
High   Control      299.630653
       Test         337.230698
Low    Control      287.518430
       Test         319.267560
Name: duration_sec, dtype: float64

Segment: clnt_age  |  Metric: went_backward
p-value: 0.00000
group 

In [37]:

# Ask Claude to write the plain-English explanation

prompt = f"""
You are a data analyst assistant. Below are statistical test results comparing
a Test group vs Control group across several segments.

IMPORTANT: Before ranking any finding as important, check whether both Control
and Test have real data for that group. If a group shows exactly 100% or 0%,
or is missing one side of the comparison entirely, treat it as unreliable due
to a likely small sample size — do NOT rank it as a top finding. Mention it
separately as "unreliable due to insufficient data."

For each segment, explain in plain English:
1. Whether the difference is statistically significant (p-value below 0.05)
2. Which specific group shows the strongest effect (excluding unreliable ones)
3. Rank the segments from most to least important finding

Here is the data:
{findings_text}
"""

response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1000,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.content[0].text)

# Statistical Test Results Analysis

## Overview
All segments show **statistically significant differences** (p-value < 0.00000), meaning the Test group performs differently from Control across all measures. All groups have realistic percentages/values, indicating sufficient sample sizes.

---

## Key Findings by Metric

### **1. COMPLETION RATE (completed)**
- **Test group shows HIGHER completion rates** across all segments
- **Low Age group**: Strongest positive effect
  - Control: 67.4% → Test: 71.5% (+4.1 percentage points)
- **Low Tenure group**: Second strongest
  - Control: 66.2% → Test: 70.1% (+3.9 percentage points)
- **High Tenure**: +3.5 percentage points
- **High Age**: +3.2 percentage points

### **2. DURATION (duration_sec)**
- **Test group shows LONGER durations** across all segments
- **High Age group**: Strongest effect
  - Control: 334 sec → Test: 381 sec (+47 seconds, +14% increase)
- **High Tenure**: +38 seconds (+13%)
- **Low Tenure**: +32 seconds (+11%)
- **Low Ag

In [39]:


# Ask Claude to explain the findings like an analyst would

prompt = f"""
You are a data analyst assistant. Below are statistical test results comparing
a Test group vs Control group across several segments, for THREE different metrics:
completion rate, average process duration, and backward navigation rate.

IMPORTANT RULES:
1. If a group shows exactly 100% or 0%, or is missing one side of the comparison,
   treat it as unreliable due to small sample size — do not use it in your top findings.
2. Do NOT recommend targeting specific demographic groups (age, gender, tenure) —
   only describe the pattern found in the data. Targeting decisions are a business
   judgment call, not something to state as a recommendation.
3. Specifically check: does the completion rate improvement for the Test group come
   with a cost in duration or backward navigation? Call this out clearly if so —
   this is the most important thing to look for.
4. STRICTLY describe WHAT the data shows, not WHY it happens. Do not guess at user
   motivations, comfort levels, confusion, or psychological reasons behind the numbers
   (for example, do not say things like "users find this harder" or "less comfortable
   with this approach"). Only state the measured facts: which group, which metric,
   what the size of the difference is, and whether it is statistically significant.
   Any explanation of WHY a pattern exists should be phrased as "this data does not
   explain why — it would need further investigation" rather than a guessed reason.

Here is the data:
{findings_text}
"""

response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=1500,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response.content[0].text)





# Statistical Test Results Analysis: Test vs Control

## Key Finding: Trade-offs Accompanying Completion Rate Improvement

**The Test group shows higher completion rates across all segments, BUT this improvement comes with measurable costs in both duration and backward navigation.**

### Overall Pattern Across All Segments

| Metric | Test vs Control | Magnitude |
|--------|----------------|-----------|
| **Completion Rate** | Higher in Test | +3.2 to +4.1 percentage points |
| **Process Duration** | Higher in Test | +24 to +47 seconds longer |
| **Backward Navigation Rate** | Higher in Test | +4.2 to +9.5 percentage points |

All differences are statistically significant (p < 0.00001).

---

## Detailed Findings by Metric

### 1. Completion Rate
Test group shows consistently higher completion rates:
- **High age**: 67.1% (Test) vs 63.8% (Control) — **+3.2 pts**
- **Low age**: 71.5% (Test) vs 67.4% (Control) — **+4.1 pts**
- **High tenure**: 68.5% (Test) vs 65.0% (Control) — **+3.5 pts

In [40]:

# Save the agent's explanation to a file


with open("agent_findings.txt", "w") as f:
    f.write(response.content[0].text)

print("Saved agent's explanation to agent_findings.txt")

Saved agent's explanation to agent_findings.txt


In [ ]:
## Summary of findings — Automated Agent Investigation

**Bugs found and fixed during development:**
1. First agent run: sample-size guardrail checked the wrong column (raw age/tenure values instead of the grouped High/Low buckets), causing every segment to be incorrectly skipped as "too small."
2. After the fix, the guardrail correctly caught two genuinely unreliable segments — `gendr` (X group: 2 people) and `num_accts` (Low group: 14 people) — and skipped them rather than reporting misleading 100%/0% results.

**Reliable segments tested:** `clnt_age` and `clnt_tenure_yr`, across three metrics: completion rate, duration, and backward navigation.

**Agent's findings — do they match the manual baseline from notebook 02?**

| Metric | Manual finding (02) | Agent finding (03) | Match? |
|---|---|---|---|
| Completion rate | Test higher (65.6% → 69.3%) | Test higher across all age/tenure groups (+3.2 to +4.1pp) | ✅ |
| Duration | Test +12% longer (293.8s → 328.4s) | Test +10–14% longer across all groups | ✅ |
| Backward navigation | Test +7.3pp higher (26.1% → 33.4%) | Test +4.2 to +10.5pp higher, concentrated most in High Age/High Tenure | ✅ (and more specific) |

**New insight the agent surfaced beyond the manual analysis:**
The friction cost (backward navigation, duration) is not evenly distributed — it is noticeably larger for **High Age** (+10.5pp backward navigation, +14% duration) and **High Tenure** (+9.5pp, +13% duration) clients, compared to Low Age/Low Tenure clients. The manual analysis found segment cuts existed but didn't explicitly connect the friction *cost* to specific segments — the agent's automated cross-testing of all metrics × all segments surfaced this connection directly.

**Conclusion:** The agent successfully and independently reproduced the core manual finding — that the Test group's completion-rate win comes with a real friction cost — and added a layer of segment-specific nuance that wasn't explicit in the original manual pass. This validates the core premise of the project: an automated, guardrailed investigation can match (and in this case extend) manual analyst work, provided sample-size validation is in place to prevent false findings.

**Caveat:** The agent's phrasing occasionally leans toward interpreting *why* a group struggles (e.g. "less comfortable with this approach") rather than strictly describing the pattern. This is a prompt-engineering issue to refine further — the underlying statistical findings are sound, but the narrative language needs tightening to stay strictly descriptive rather than interpretive.